In [1]:
import requests
import os

import pandas as pd
import numpy as np

import json
from tqdm.autonotebook import tqdm

import dask.dataframe as dd
from dask.multiprocessing import get
from dask.diagnostics import ProgressBar

from datetime import datetime

from IPython.display import display

tqdm.pandas()

/tmp/ipykernel_92803/3356625247.py:8: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from tqdm.autonotebook import tqdm


In [ ]:
# Potential improvement: strip + upper to reduce the number of unique addresses
# For MW: ~3% less addresses, but cannot be done inplace, otherwise reconciliation wont work.

# Parameters

In [2]:
# Field names required by bePelias. Do not change!
street_field  = "streetName"
housenbr_field = "houseNumber"
postcode_field = "postCode"
city_field  =    "postName"
country_field =  "countryName"


In [3]:
# IP:port of bePelias instance
ws_hostname = "172.27.0.64:4001"

# Parameter to update to your dataset --> replace keys by your input file column names

field_mapping = {
    "streetName": street_field,
    "houseNumber": housenbr_field,
    "postCode": postcode_field,
    "municipalityName": city_field}

data_dir="data/geocoding/mw2"
# data_dir="data/geocoding/bepelias_batch/"
data_dir="data/geocoding/mw2_20240913/"


input_filename = "addresses_mw2.csv.gz" # Has to be a csv or csv.gz file
input_filename = "addresses.csv" # Has to be a csv or csv.gz file
# input_filename = "sample.csv.gz"

# If you only want to geocode a part of the file, add a query filter (sent to "query()" pandas function)
# filter_query = "countryNisCode.isnull() | (countryNisCode==150)"
filter_query = "countryNisCode.isnull() | (countryNisCode=='150.0')"


# Applyed just before data is sent to geocoded, on the whole dataframe
replace_dict = {"unknown": ""}

In [4]:
# addresses.replace(replace)

In [5]:
cache_filename = "cache.pkl.gz" # Will write bePelias result every "cache_chunk_size" record. Allow to not restart from scratch in case of crash
cache_chunk_size = 10000

In [6]:
output_filename = input_filename.split(".", maxsplit=1)
output_filename = f'{output_filename[0]}_output.{output_filename[1]}'
print(f"Output filename: {output_filename}")

Output filename: addresses_output.csv


# Functions

In [7]:
def call_ws(addr_data, mode="advanced"): #lg = "en,fr,nl"
    t = datetime.now()
    
    fields = { 
            "mode": mode
        }

    if isinstance(addr_data, pd.Series):
        addr_data = addr_data.to_dict()
        
    try: 
        r = requests.get(
        f'http://{ws_hostname}/REST/bepelias/v1/geocode',
            params=addr_data)
        

    except Exception as e:
        print("Exception !")
        print(addr_data)
        print(e)
        raise e
        
    if r.status_code == 204:
        # print("No result!")
        # print(addr_data)
        # print(r.text)
        return
    elif r.status_code == 400:
        print("Argument error")
        print(r.text)
    elif r.status_code == 200:
        try:
            res = json.loads(r.text)
            res["time"] = (datetime.now() - t).total_seconds()
        except ValueError as ve:

            print("Cannot decode result:")
            print(ve)
            print(r.text)
            return r.text
        except AttributeError as ae:
            print(ae)
            print(type(r.text))
            print(r.text)
        return res
    else: 
        print(f"Unknown return code: {r.status_code} ")
        print(r.text)



# Calls

## Read data

In [8]:
addresses_filename = f"{data_dir}/{input_filename}"
addresses = pd.read_csv(addresses_filename, dtype=str)
addresses

,streetName,juridicalDistrict,addressType,houseNumber,postCode,id,municipalityName,countryNisCode,targetId,boxNumber
0,Rue Saint-Hubert(DV),910,legal,84,5100,416afc98-9bc4-46a7-b76d-a24295a695d2,Namur,150.0,V0It0ooBL0HW2X5F5T_J,NaN
1,Rue Saint-Hubert(DV),NaN,establishmentUnit,84,5100,2a0099c0-ad23-4afc-acca-e12f9d02027f,Namur,NaN,V0It0ooBL0HW2X5F5T_J,NaN
2,Pladijsstraat,520,legal,2,8540,7195703d-7c33-4c91-abd1-30c8e15e49d6,Deerlijk,150.0,muT30YoBc6EOG49bA0Mq,NaN
3,Vichtestraat(O),NaN,establishmentUnit,31A,8553,2e0ca445-a01e-4386-96e1-17a451644394,Zwevegem,NaN,muT30YoBc6EOG49bA0Mq,NaN
4,Pladijsstraat,NaN,establishmentUnit,2,8540,b11e9038-3a48-4aae-9b11-7ecfc7d982cd,Deerlijk,NaN,muT30YoBc6EOG49bA0Mq,NaN
...,...,...,...,...,...,...,...,...,...,...
1398632,Place Jean Absil(B.S.),NaN,establishmentUnit,10,7603,4ffec516-c350-447c-80c6-b65b83fbb6a5,Péruwelz,NaN,wZhd0ooBUOVbPdvwYZPD,NaN
1398633,Meensesteenweg,520,legal,714,8800,e942e7d9-57a6-4f0c-afae-4f15c0345a1a,Roeselare,150.0,5uRZ1IoBc6EOG49bItVN,NaN
1398634,Meensesteenweg,NaN,establishmentUnit,389,8800,d90f2cf9-8028-4ee6-89f1-ec56d097cccc,Roeselare,NaN,5uRZ1IoBc6EOG49bItVN,NaN
1398635,Avenue Sergent Vrithoff,910,legal,129,5000,e717f0cb-be91-4da0-91de-a4fa82b08265,Namur,150.0,x-WO1YoBc6EOG49byh6j,4


In [10]:
print(f"Input dataset: {addresses.shape[0]} rows")

Input dataset: 1398637 rows


In [11]:
if filter_query:
    addresses = addresses.query(filter_query)
    print(f"After filtering: {addresses.shape[0]} rows")
addresses    

After filtering: 1391302 rows


,streetName,juridicalDistrict,addressType,houseNumber,postCode,id,municipalityName,countryNisCode,targetId,boxNumber
0,Rue Saint-Hubert(DV),910,legal,84,5100,416afc98-9bc4-46a7-b76d-a24295a695d2,Namur,150.0,V0It0ooBL0HW2X5F5T_J,NaN
1,Rue Saint-Hubert(DV),NaN,establishmentUnit,84,5100,2a0099c0-ad23-4afc-acca-e12f9d02027f,Namur,NaN,V0It0ooBL0HW2X5F5T_J,NaN
2,Pladijsstraat,520,legal,2,8540,7195703d-7c33-4c91-abd1-30c8e15e49d6,Deerlijk,150.0,muT30YoBc6EOG49bA0Mq,NaN
3,Vichtestraat(O),NaN,establishmentUnit,31A,8553,2e0ca445-a01e-4386-96e1-17a451644394,Zwevegem,NaN,muT30YoBc6EOG49bA0Mq,NaN
4,Pladijsstraat,NaN,establishmentUnit,2,8540,b11e9038-3a48-4aae-9b11-7ecfc7d982cd,Deerlijk,NaN,muT30YoBc6EOG49bA0Mq,NaN
...,...,...,...,...,...,...,...,...,...,...
1398632,Place Jean Absil(B.S.),NaN,establishmentUnit,10,7603,4ffec516-c350-447c-80c6-b65b83fbb6a5,Péruwelz,NaN,wZhd0ooBUOVbPdvwYZPD,NaN
1398633,Meensesteenweg,520,legal,714,8800,e942e7d9-57a6-4f0c-afae-4f15c0345a1a,Roeselare,150.0,5uRZ1IoBc6EOG49bItVN,NaN
1398634,Meensesteenweg,NaN,establishmentUnit,389,8800,d90f2cf9-8028-4ee6-89f1-ec56d097cccc,Roeselare,NaN,5uRZ1IoBc6EOG49bItVN,NaN
1398635,Avenue Sergent Vrithoff,910,legal,129,5000,e717f0cb-be91-4da0-91de-a4fa82b08265,Namur,150.0,x-WO1YoBc6EOG49byh6j,4


In [26]:
addresses_unique = addresses[field_mapping.keys()].drop_duplicates()
print(f"Unique addresses : {addresses_unique.shape[0]} rows")
addresses_unique

Unique addresses : 439977 rows


,streetName,houseNumber,postCode,municipalityName
0,RUE SAINT-HUBERT(DV),84,5100,NAMUR
2,PLADIJSSTRAAT,2,8540,DEERLIJK
3,VICHTESTRAAT(O),31A,8553,ZWEVEGEM
5,HEIDEBAAN,90,9100,SINT-NIKLAAS
7,JULES MALOULAAN,26,1040,ETTERBEEK
...,...,...,...,...
1398630,HOEKSTRAAT,34,9260,WICHELEN
1398631,PLACE JEAN ABSIL(B.S.),10,7603,PÉRUWELZ
1398633,MEENSESTEENWEG,714,8800,ROESELARE
1398634,MEENSESTEENWEG,389,8800,ROESELARE


In [13]:
try: 
    addresses_geocoded = pd.read_pickle(f"{data_dir}/{cache_filename}")
except FileNotFoundError:
    addresses_geocoded = pd.DataFrame(columns =addresses_unique.columns )
    addresses_geocoded["json"]=pd.NA
addresses_geocoded   

,streetName,houseNumber,postCode,municipalityName,json
0,Rue Saint-Hubert(DV),84,5100,Namur,"{'geocoding': {'version': '0.2', 'attribution'..."
1,Pladijsstraat,2,8540,Deerlijk,"{'geocoding': {'version': '0.2', 'attribution'..."
2,Vichtestraat(O),31A,8553,Zwevegem,"{'geocoding': {'version': '0.2', 'attribution'..."
3,Heidebaan,90,9100,Sint-Niklaas,"{'geocoding': {'version': '0.2', 'attribution'..."
4,Jules Maloulaan,26,1040,Etterbeek,"{'geocoding': {'version': '0.2', 'attribution'..."
...,...,...,...,...,...
452124,Hoekstraat,34,9260,Wichelen,"{'geocoding': {'version': '0.2', 'attribution'..."
452125,Place Jean Absil(B.S.),10,7603,Péruwelz,"{'geocoding': {'version': '0.2', 'attribution'..."
452126,Meensesteenweg,714,8800,Roeselare,"{'geocoding': {'version': '0.2', 'attribution'..."
452127,Meensesteenweg,389,8800,Roeselare,"{'geocoding': {'version': '0.2', 'attribution'..."


In [14]:
addresses_to_geocode = addresses_unique.merge(addresses_geocoded, indicator=True, how="left")
print(f"Found {addresses_to_geocode[addresses_to_geocode._merge == 'both'].shape[0]} addresses in cache")
addresses_to_geocode = addresses_to_geocode[addresses_to_geocode._merge =='left_only']
addresses_to_geocode = addresses_to_geocode.drop(columns=["json", "_merge"])
addresses_to_geocode

Found 452129 addresses in cache


,streetName,houseNumber,postCode,municipalityName


In [15]:
chunks = [addresses_to_geocode.iloc[i:i+cache_chunk_size] for i in range(0,len(addresses_to_geocode),cache_chunk_size)]
print(f"{len(chunks)} chunks of size {cache_chunk_size}")


0 chunks of size 10000


# Geocode

In [16]:
initial_start = datetime.now()
for chunk in tqdm(chunks):
    chunk_start = datetime.now()
    dd_addresses = dd.from_pandas(chunk.replace(replace_dict).rename(columns=field_mapping).fillna(""), 
                                  npartitions=min(64, chunk.shape[0]))

    dask_task = dd_addresses[[street_field, housenbr_field, postcode_field, city_field]].apply(call_ws, meta=('x', 'str'), axis=1)

    with ProgressBar(): 
        chunk["json"] = dask_task.compute()


    chunk_time = (datetime.now() - chunk_start).total_seconds()

    ips=chunk.shape[0]/chunk_time

    print(f"{chunk_time:.2f} seconds, {ips:.2f} it/s, {ips*3600:.0f} it/h")

    addresses_geocoded = pd.concat([addresses_geocoded, chunk]).drop_duplicates(subset=addresses_geocoded.drop("json", axis=1).columns)
    
    # To reduce the risk of loosing cache of previous blocs if process crashes during write, 
    # we write a temp file (_cache.pkl.gz) and move it to the final file (cache.pkl.gz) afterward
    addresses_geocoded.to_pickle(f"{data_dir}/_{cache_filename}")
    os.rename(f"{data_dir}/_{cache_filename}", f"{data_dir}/{cache_filename}")
    
    
    

total_time = (datetime.now() - initial_start).total_seconds()

ips=addresses_to_geocode.shape[0]/total_time

print(f"Global : {total_time:.2f} seconds, {ips:.2f} it/s, {ips*3600:.0f} it/h")


0it [00:00, ?it/s]

Global : 0.00 seconds, 0.00 it/s, 0 it/h


In [17]:
# os.rename(f"{data_dir}/_{cache_filename}", f"{data_dir}/{cache_filename}")

In [18]:
addresses_geocoded#.drop_duplicates(subset=addresses_geocoded.drop(["json", "geom"], axis=1).columns)

,streetName,houseNumber,postCode,municipalityName,json
0,Rue Saint-Hubert(DV),84,5100,Namur,"{'geocoding': {'version': '0.2', 'attribution'..."
1,Pladijsstraat,2,8540,Deerlijk,"{'geocoding': {'version': '0.2', 'attribution'..."
2,Vichtestraat(O),31A,8553,Zwevegem,"{'geocoding': {'version': '0.2', 'attribution'..."
3,Heidebaan,90,9100,Sint-Niklaas,"{'geocoding': {'version': '0.2', 'attribution'..."
4,Jules Maloulaan,26,1040,Etterbeek,"{'geocoding': {'version': '0.2', 'attribution'..."
...,...,...,...,...,...
452124,Hoekstraat,34,9260,Wichelen,"{'geocoding': {'version': '0.2', 'attribution'..."
452125,Place Jean Absil(B.S.),10,7603,Péruwelz,"{'geocoding': {'version': '0.2', 'attribution'..."
452126,Meensesteenweg,714,8800,Roeselare,"{'geocoding': {'version': '0.2', 'attribution'..."
452127,Meensesteenweg,389,8800,Roeselare,"{'geocoding': {'version': '0.2', 'attribution'..."


# Reconciliate

In [19]:
def get(dct, keys):
    """
    Get an item of "dct" (dict), by going through "keys", or None 
    Example : 
    keys = ["a", "b", "c"] ==> will return dct["a"]["b"]["c"], or None in any key of the sequence does not exists

    Parameters
    ----------
    dct : dict
        Dictionnary
    keys : list

    Returns
    -------
    An element of dct or None.
    """
    
    for k in keys:
        try: 
            if  dct is None:
                return None
            dct = dct[k]
        except KeyError:
            return None
    return dct

In [21]:
output = addresses_geocoded.copy()
output["best_id"]   = output.json.apply(lambda r: get(r, ["features", 0, "properties", 'addendum', 'best', 'best_id']) or get(r, ["features", 0, "properties", 'addendum', 'best', 'street_id']))
output["geom"]      = output.json.apply(lambda r: get(r, ["features", 0, "geometry","coordinates"] ))
output["precision"] = output.json.apply(lambda r: get(r, ["bepelias", "precision"]) or  "[none]")


output = addresses.merge(output.drop(columns="json"))

output.to_csv(f"{data_dir}/{output_filename}", index=False)


In [22]:
output.to_csv(f"{data_dir}/{output_filename}.gz", index=False)

In [23]:
def print_precision_stats(df):
    vc = df["precision"].value_counts()

    with pd.option_context("display.float_format", '{:,.2%}'.format):
        print(vc/df.shape[0])
    
    print("")
    print(f'building:  {vc[["address", "street_interpol", "address_interpol"]].sum()/df.shape[0]:.2%}')
    print(f'street:    {vc[["street", "address_streetcenter"]].sum()/df.shape[0]:.2%}')
    print(f'city:      {vc[["city"]].sum()/df.shape[0]:.2%}')
    print(f'country:   {vc[[f for f in ["street_00", "address_00", "country"] if f in vc ]].sum()/df.shape[0]:.2%}')
    print(f'NONE:      {vc[["[none]"]].sum()/df.shape[0]:.2%}')

In [24]:
print("Precision stats for all addresses")
print_precision_stats(output)

Precision stats for all addresses
precision
address                78.22%
street                  8.92%
city                    7.85%
street_interpol         3.62%
address_streetcenter    0.54%
address_interpol        0.35%
street_00               0.27%
address_00              0.15%
[none]                  0.08%
Name: count, dtype: float64

building:  82.20%
street:    9.46%
city:      7.85%
country:   0.42%
NONE:      0.08%


In [25]:
print("Precision stats per unique addresses")
print_precision_stats(output.drop_duplicates(subset=field_mapping.keys()))

Precision stats per unique addresses
precision
address                86.82%
street                  4.69%
street_interpol         4.03%
city                    3.01%
address_streetcenter    0.59%
address_interpol        0.49%
street_00               0.18%
address_00              0.15%
[none]                  0.04%
Name: count, dtype: float64

building:  91.34%
street:    5.28%
city:      3.01%
country:   0.33%
NONE:      0.04%


In [26]:
addresses_geocoded[addresses_geocoded.json.isnull()]

,streetName,houseNumber,postCode,municipalityName,json
1601,NaN,5,1600,NaN,None
2002,Tramlijn 10 - 10848W,unknown,1120,Neder-over-Heembeek (Bru.),None
5067,NaN,13,6141,NaN,None
5595,unknown,unknown,1035,Minis Rég. de Brux.-Capitale,None
9994,unknown,unknown,4681,Hermalle-sous-Argenteau,None
...,...,...,...,...,...
438089,unknown,unknown,4577,Vierset-Barse,None
441901,NaN,143,9200,NaN,None
444257,NaN,7,8300,NaN,None
445994,NaN,25B,3140,NaN,None


In [27]:
# addresses_dask.iloc[0].json
# addresses_dask[addresses_dask.precision == "no_feat"]
addresses_geocoded[addresses_geocoded.json.isnull()]

,streetName,houseNumber,postCode,municipalityName,json
1601,NaN,5,1600,NaN,None
2002,Tramlijn 10 - 10848W,unknown,1120,Neder-over-Heembeek (Bru.),None
5067,NaN,13,6141,NaN,None
5595,unknown,unknown,1035,Minis Rég. de Brux.-Capitale,None
9994,unknown,unknown,4681,Hermalle-sous-Argenteau,None
...,...,...,...,...,...
438089,unknown,unknown,4577,Vierset-Barse,None
441901,NaN,143,9200,NaN,None
444257,NaN,7,8300,NaN,None
445994,NaN,25B,3140,NaN,None


In [28]:
# cach = pd.read_pickle("data/geocoding/mw2/_cache.pkl.gz")

In [29]:
# cach[cach.streetName=="Westbekesluis"]

# tests

In [5]:
import geopandas as gpd
import pandas as pd

In [26]:
pd.read_csv("../bePelias/data/bestaddresses_bewal.csv", nrows=10)

,id,lat,lon,housenumber,locality,street,postalcode,source,name,name_fr,name_nl,name_de,layer,country,addendum_json_best
0,geodata.wallonie.be/id/Address/1871131/2_fr,50.67028,5.54013,172,Liège,Rue d'Ans,4000,BE-WAL,"172, Rue d'Ans, 4000 Liège","172, Rue d'Ans, 4000 Liège",NaN,NaN,address,Belgium,"{""best_id"": ""geodata.wallonie.be/id/Address/18..."
1,geodata.wallonie.be/id/Address/1871257/2_fr,50.64810,5.55130,374,Liège,Rue Sainte-Marguerite,4000,BE-WAL,"374, Rue Sainte-Marguerite, 4000 Liège","374, Rue Sainte-Marguerite, 4000 Liège",NaN,NaN,address,Belgium,"{""best_id"": ""geodata.wallonie.be/id/Address/18..."
2,geodata.wallonie.be/id/Address/1870940/2_fr,50.64125,5.56686,136B,Liège,Boulevard de la Sauvenière,4000,BE-WAL,"136B, Boulevard de la Sauvenière, 4000 Liège","136B, Boulevard de la Sauvenière, 4000 Liège",NaN,NaN,address,Belgium,"{""best_id"": ""geodata.wallonie.be/id/Address/18..."
3,geodata.wallonie.be/id/Address/1871089/3_fr,50.63195,5.63372,23,Liège,Rue des Noctuelles,4020,BE-WAL,"23, Rue des Noctuelles, 4020 Liège","23, Rue des Noctuelles, 4020 Liège",NaN,NaN,address,Belgium,"{""best_id"": ""geodata.wallonie.be/id/Address/18..."
4,geodata.wallonie.be/id/Address/1871032/2_fr,50.61461,5.53652,376,Liège,Rue Ernest Solvay,4000,BE-WAL,"376, Rue Ernest Solvay, 4000 Liège","376, Rue Ernest Solvay, 4000 Liège",NaN,NaN,address,Belgium,"{""best_id"": ""geodata.wallonie.be/id/Address/18..."
5,geodata.wallonie.be/id/Address/1871282/2_fr,50.64943,5.56377,1,Liège,Rue Isi Collin,4000,BE-WAL,"1, Rue Isi Collin, 4000 Liège","1, Rue Isi Collin, 4000 Liège",NaN,NaN,address,Belgium,"{""best_id"": ""geodata.wallonie.be/id/Address/18..."
6,geodata.wallonie.be/id/Address/1872703/2_fr,50.64104,5.62427,110B,Liège,Rue de Bois-de-Breux,4020,BE-WAL,"110B, Rue de Bois-de-Breux, 4020 Liège","110B, Rue de Bois-de-Breux, 4020 Liège",NaN,NaN,address,Belgium,"{""best_id"": ""geodata.wallonie.be/id/Address/18..."
7,geodata.wallonie.be/id/Address/1874425/2_fr,50.65767,5.57330,95,Liège,Rue des Tawes,4000,BE-WAL,"95, Rue des Tawes, 4000 Liège","95, Rue des Tawes, 4000 Liège",NaN,NaN,address,Belgium,"{""best_id"": ""geodata.wallonie.be/id/Address/18..."
8,geodata.wallonie.be/id/Address/1874451/2_fr,50.65727,5.59918,150,Liège,Boulevard Ernest Solvay,4000,BE-WAL,"150, Boulevard Ernest Solvay, 4000 Liège","150, Boulevard Ernest Solvay, 4000 Liège",NaN,NaN,address,Belgium,"{""best_id"": ""geodata.wallonie.be/id/Address/18..."
9,geodata.wallonie.be/id/Address/1874453/2_fr,50.66177,5.59558,539,Liège,Boulevard Ernest Solvay,4000,BE-WAL,"539, Boulevard Ernest Solvay, 4000 Liège","539, Boulevard Ernest Solvay, 4000 Liège",NaN,NaN,address,Belgium,"{""best_id"": ""geodata.wallonie.be/id/Address/18..."


In [27]:
data = pd.read_csv("../bePelias/data/bestaddresses_bewal.csv", usecols=["id", "lat", "lon", "postalcode"])
data

,id,lat,lon,postalcode
0,geodata.wallonie.be/id/Address/1871131/2_fr,50.67028,5.54013,4000
1,geodata.wallonie.be/id/Address/1871257/2_fr,50.64810,5.55130,4000
2,geodata.wallonie.be/id/Address/1870940/2_fr,50.64125,5.56686,4000
3,geodata.wallonie.be/id/Address/1871089/3_fr,50.63195,5.63372,4020
4,geodata.wallonie.be/id/Address/1871032/2_fr,50.61461,5.53652,4000
...,...,...,...,...
1615541,geodata.wallonie.be/id/Address/1452688/1_de,50.22558,6.08834,4790
1615542,geodata.wallonie.be/id/Address/1876520/2_de,50.61820,6.04550,4700
1615543,geodata.wallonie.be/id/Address/1997439/1_de,0.00000,0.00000,4700
1615544,geodata.wallonie.be/id/Address/1997438/1_de,0.00000,0.00000,4700


In [24]:
data[data.lon.between(3.05, 3.08) ].sort_values("lat") #3.075001, lat=48.247268

,id,lat,lon


In [30]:
street_res = {'geocoding': {'version': '0.2', 'attribution': 'http://172.27.0.64:4000/attribution', 'query': {'parsed_text': {'postalcode': '4453', 'street': 'Chaussée Brunehault'}, 'size': 10, 'private': False, 'lang': {'name': 'English', 'iso6391': 'en', 'iso6393': 'eng', 'via': 'default', 'defaulted': True}, 'querySize': 20}, 'engine': {'name': 'Pelias', 'author': 'Mapzen', 'version': '1.0'}, 'timestamp': 1725631299436}, 'type': 'FeatureCollection', 'features': [{'type': 'Feature', 'geometry': {'type': 'Point', 'coordinates': [4.253754, 50.453838]}, 'properties': {'id': 'geodata.wallonie.be/id/Streetname/7721609/2_fr', 'gid': 'be-wal:street:geodata.wallonie.be/id/Streetname/7721609/2_fr', 'layer': 'street', 'source': 'be-wal', 'source_id': 'geodata.wallonie.be/id/Streetname/7721609/2_fr', 'country_code': 'BE', 'name': 'Chaussée Brunehault, 7141 Morlanwelz', 'street': 'Chaussée Brunehault', 'postalcode': '7141', 'confidence': 1, 'match_type': 'exact', 'accuracy': 'centroid', 'country': 'Belgium', 'country_gid': 'whosonfirst:country:85632997', 'country_a': 'BEL', 'macroregion': 'Wallonia', 'macroregion_gid': 'whosonfirst:macroregion:404227353', 'region': 'Hainaut', 'region_gid': 'whosonfirst:region:85681723', 'region_a': 'HT', 'county': 'Thuin', 'county_gid': 'whosonfirst:county:102049923', 'county_a': 'TN', 'locality': 'Morlanwelz-Mariemont', 'locality_gid': 'whosonfirst:locality:101755011', 'label': 'Chaussée Brunehault, 7141 Morlanwelz, Morlanwelz-Mariemont, HT, Belgium', 'addendum': {'best': {'streetname_fr': 'Chaussée Brunehault', 'municipality_name_fr': 'Morlanwelz', 'NIS': 58004, 'municipality_id': 'geodata.wallonie.be/id/Municipality/58004/7', 'street_id': 'geodata.wallonie.be/id/Streetname/7721609/2'}}}}, {'type': 'Feature', 'geometry': {'type': 'Point', 'coordinates': [4.275132, 50.280646]}, 'properties': {'id': 'geodata.wallonie.be/id/Streetname/7719225/1_fr', 'gid': 'be-wal:street:geodata.wallonie.be/id/Streetname/7719225/1_fr', 'layer': 'street', 'source': 'be-wal', 'source_id': 'geodata.wallonie.be/id/Streetname/7719225/1_fr', 'country_code': 'BE', 'name': 'Chaussée Brunehault, 6511 Beaumont', 'street': 'Chaussée Brunehault', 'postalcode': '6511', 'confidence': 1, 'match_type': 'exact', 'accuracy': 'centroid', 'country': 'Belgium', 'country_gid': 'whosonfirst:country:85632997', 'country_a': 'BEL', 'macroregion': 'Wallonia', 'macroregion_gid': 'whosonfirst:macroregion:404227353', 'region': 'Hainaut', 'region_gid': 'whosonfirst:region:85681723', 'region_a': 'HT', 'county': 'Thuin', 'county_gid': 'whosonfirst:county:102049923', 'county_a': 'TN', 'label': 'Chaussée Brunehault, 6511 Beaumont, HT, Belgium', 'addendum': {'best': {'streetname_fr': 'Chaussée Brunehault', 'municipality_name_fr': 'Beaumont', 'NIS': 56005, 'municipality_id': 'geodata.wallonie.be/id/Municipality/56005/7', 'street_id': 'geodata.wallonie.be/id/Streetname/7719225/1'}}}}, {'type': 'Feature', 'geometry': {'type': 'Point', 'coordinates': [4.191175, 50.430885]}, 'properties': {'id': 'geodata.wallonie.be/id/Streetname/7719493/2_fr', 'gid': 'be-wal:street:geodata.wallonie.be/id/Streetname/7719493/2_fr', 'layer': 'street', 'source': 'be-wal', 'source_id': 'geodata.wallonie.be/id/Streetname/7719493/2_fr', 'country_code': 'BE', 'name': 'Chaussée Brunehault, 7134 Binche', 'street': 'Chaussée Brunehault', 'postalcode': '7134', 'confidence': 1, 'match_type': 'exact', 'accuracy': 'centroid', 'country': 'Belgium', 'country_gid': 'whosonfirst:country:85632997', 'country_a': 'BEL', 'macroregion': 'Wallonia', 'macroregion_gid': 'whosonfirst:macroregion:404227353', 'region': 'Hainaut', 'region_gid': 'whosonfirst:region:85681723', 'region_a': 'HT', 'county': 'Thuin', 'county_gid': 'whosonfirst:county:102049923', 'county_a': 'TN', 'locality': 'Ressaix', 'locality_gid': 'whosonfirst:locality:101755029', 'label': 'Chaussée Brunehault, 7134 Binche, Ressaix, HT, Belgium', 'addendum': {'best': {'streetname_fr': 'Chaussée Brunehault', 'municipality_name_fr': 'Binche', 'NIS': 58002, 'municipality_id': 'geodata.wallonie.be/id/Municipality/58002/7', 'street_id': 'geodata.wallonie.be/id/Streetname/7719493/2'}}}}, {'type': 'Feature', 'geometry': {'type': 'Point', 'coordinates': [4.041498, 50.667605]}, 'properties': {'id': 'geodata.wallonie.be/id/Streetname/7718277/2_fr', 'gid': 'be-wal:street:geodata.wallonie.be/id/Streetname/7718277/2_fr', 'layer': 'street', 'source': 'be-wal', 'source_id': 'geodata.wallonie.be/id/Streetname/7718277/2_fr', 'country_code': 'BE', 'name': 'Chaussée Brunehault, 7830 Silly', 'street': 'Chaussée Brunehault', 'postalcode': '7830', 'confidence': 1, 'match_type': 'exact', 'accuracy': 'centroid', 'country': 'Belgium', 'country_gid': 'whosonfirst:country:85632997', 'country_a': 'BEL', 'macroregion': 'Wallonia', 'macroregion_gid': 'whosonfirst:macroregion:404227353', 'region': 'Hainaut', 'region_gid': 'whosonfirst:region:85681723', 'region_a': 'HT', 'county': 'Soignies', 'county_gid': 'whosonfirst:county:102049943', 'county_a': 'SG', 'locality': 'Enghien', 'locality_gid': 'whosonfirst:locality:101838653', 'label': 'Chaussée Brunehault, 7830 Silly, Enghien, HT, Belgium', 'addendum': {'best': {'streetname_fr': 'Chaussée Brunehault', 'municipality_name_fr': 'Silly', 'NIS': 51068, 'municipality_id': 'geodata.wallonie.be/id/Municipality/51068/7', 'street_id': 'geodata.wallonie.be/id/Streetname/7718277/2'}}}}, {'type': 'Feature', 'geometry': {'type': 'Point', 'coordinates': [3.691458, 50.58684]}, 'properties': {'id': 'geodata.wallonie.be/id/Streetname/7724152/1_fr', 'gid': 'be-wal:street:geodata.wallonie.be/id/Streetname/7724152/1_fr', 'layer': 'street', 'source': 'be-wal', 'source_id': 'geodata.wallonie.be/id/Streetname/7724152/1_fr', 'country_code': 'BE', 'name': 'Chaussée Brunehault, 7903 Leuze-en-Hainaut', 'street': 'Chaussée Brunehault', 'postalcode': '7903', 'confidence': 1, 'match_type': 'exact', 'accuracy': 'centroid', 'country': 'Belgium', 'country_gid': 'whosonfirst:country:85632997', 'country_a': 'BEL', 'macroregion': 'Wallonia', 'macroregion_gid': 'whosonfirst:macroregion:404227353', 'region': 'Hainaut', 'region_gid': 'whosonfirst:region:85681723', 'region_a': 'HT', 'county': 'Tournai', 'county_gid': 'whosonfirst:county:102049951', 'county_a': 'TR', 'label': 'Chaussée Brunehault, 7903 Leuze-en-Hainaut, HT, Belgium', 'addendum': {'best': {'streetname_fr': 'Chaussée Brunehault', 'municipality_name_fr': 'Leuze-en-Hainaut', 'NIS': 57094, 'municipality_id': 'geodata.wallonie.be/id/Municipality/57094/7', 'street_id': 'geodata.wallonie.be/id/Streetname/7724152/1'}}}}, {'type': 'Feature', 'geometry': {'type': 'Point', 'coordinates': [4.063475, 50.701273]}, 'properties': {'id': 'geodata.wallonie.be/id/Streetname/7716734/5_fr', 'gid': 'be-wal:street:geodata.wallonie.be/id/Streetname/7716734/5_fr', 'layer': 'street', 'source': 'be-wal', 'source_id': 'geodata.wallonie.be/id/Streetname/7716734/5_fr', 'country_code': 'BE', 'name': 'Chaussée Brunehault, 7850 Enghien', 'street': 'Chaussée Brunehault', 'postalcode': '7850', 'confidence': 1, 'match_type': 'exact', 'accuracy': 'centroid', 'country': 'Belgium', 'country_gid': 'whosonfirst:country:85632997', 'country_a': 'BEL', 'macroregion': 'Flemish Region', 'macroregion_gid': 'whosonfirst:macroregion:404227357', 'region': 'Vlaams-Brabant', 'region_gid': 'whosonfirst:region:85681731', 'region_a': 'VB', 'county': 'Halle-Vilvoorde', 'county_gid': 'whosonfirst:county:102049973', 'county_a': 'HV', 'localadmin': 'Herne', 'localadmin_gid': 'whosonfirst:localadmin:1108830399', 'label': 'Chaussée Brunehault, 7850 Enghien, Herne, VB, Belgium', 'addendum': {'best': {'streetname_fr': 'Chaussée Brunehault', 'streetname_nl': 'Brunehaultsteenweg', 'municipality_name_fr': 'Enghien', 'municipality_name_nl': 'Edingen', 'NIS': 51067, 'municipality_id': 'geodata.wallonie.be/id/Municipality/51067/7', 'street_id': 'geodata.wallonie.be/id/Streetname/7716734/5'}}}}, {'type': 'Feature', 'geometry': {'type': 'Point', 'coordinates': [5.539947, 50.71061]}, 'properties': {'id': 'geodata.wallonie.be/id/Streetname/7729452/1_fr', 'gid': 'be-wal:street:geodata.wallonie.be/id/Streetname/7729452/1_fr', 'layer': 'street', 'source': 'be-wal', 'source_id': 'geodata.wallonie.be/id/Streetname/7729452/1_fr', 'country_code': 'BE', 'name': 'Chaussée Brunehault, 4450 Juprelle', 'street': 'Chaussée Brunehault', 'postalcode': '4450', 'confidence': 1, 'match_type': 'exact', 'accuracy': 'centroid', 'country': 'Belgium', 'country_gid': 'whosonfirst:country:85632997', 'country_a': 'BEL', 'macroregion': 'Wallonia', 'macroregion_gid': 'whosonfirst:macroregion:404227353', 'region': 'Liège', 'region_gid': 'whosonfirst:region:85681691', 'region_a': 'LG', 'county': 'Liège', 'county_gid': 'whosonfirst:county:102049941', 'county_a': 'LG', 'locality': 'Juprelle', 'locality_gid': 'whosonfirst:locality:101810469', 'label': 'Chaussée Brunehault, 4450 Juprelle, Juprelle, LG, Belgium', 'addendum': {'best': {'streetname_fr': 'Chaussée Brunehault', 'municipality_name_fr': 'Juprelle', 'NIS': 62060, 'municipality_id': 'geodata.wallonie.be/id/Municipality/62060/7', 'street_id': 'geodata.wallonie.be/id/Streetname/7729452/1'}}}}, {'type': 'Feature', 'geometry': {'type': 'Point', 'coordinates': [4.090883, 50.394643]}, 'properties': {'id': 'geodata.wallonie.be/id/Streetname/7721221/2_fr', 'gid': 'be-wal:street:geodata.wallonie.be/id/Streetname/7721221/2_fr', 'layer': 'street', 'source': 'be-wal', 'source_id': 'geodata.wallonie.be/id/Streetname/7721221/2_fr', 'country_code': 'BE', 'name': 'Chaussée Brunehault, 7120 Estinnes', 'street': 'Chaussée Brunehault', 'postalcode': '7120', 'confidence': 1, 'match_type': 'exact', 'accuracy': 'centroid', 'country': 'Belgium', 'country_gid': 'whosonfirst:country:85632997', 'country_a': 'BEL', 'macroregion': 'Wallonia', 'macroregion_gid': 'whosonfirst:macroregion:404227353', 'region': 'Hainaut', 'region_gid': 'whosonfirst:region:85681723', 'region_a': 'HT', 'county': 'Thuin', 'county_gid': 'whosonfirst:county:102049923', 'county_a': 'TN', 'locality': 'Estinnes-au-Mont', 'locality_gid': 'whosonfirst:locality:101755037', 'label': 'Chaussée Brunehault, 7120 Estinnes, Estinnes-au-Mont, HT, Belgium', 'addendum': {'best': {'streetname_fr': 'Chaussée Brunehault', 'municipality_name_fr': 'Estinnes', 'NIS': 58003, 'municipality_id': 'geodata.wallonie.be/id/Municipality/58003/7', 'street_id': 'geodata.wallonie.be/id/Streetname/7721221/2'}}}}, {'type': 'Feature', 'geometry': {'type': 'Point', 'coordinates': [3.712974, 50.390429]}, 'properties': {'id': 'geodata.wallonie.be/id/Streetname/7714480/2_fr', 'gid': 'be-wal:street:geodata.wallonie.be/id/Streetname/7714480/2_fr', 'layer': 'street', 'source': 'be-wal', 'source_id': 'geodata.wallonie.be/id/Streetname/7714480/2_fr', 'country_code': 'BE', 'name': 'Chaussée Brunehault, 7380 Quiévrain', 'street': 'Chaussée Brunehault', 'postalcode': '7380', 'confidence': 1, 'match_type': 'exact', 'accuracy': 'centroid', 'country': 'Belgium', 'country_gid': 'whosonfirst:country:85632997', 'country_a': 'BEL', 'macroregion': 'Wallonia', 'macroregion_gid': 'whosonfirst:macroregion:404227353', 'region': 'Hainaut', 'region_gid': 'whosonfirst:region:85681723', 'region_a': 'HT', 'county': 'Mons', 'county_gid': 'whosonfirst:county:102049933', 'county_a': 'MN', 'label': 'Chaussée Brunehault, 7380 Quiévrain, HT, Belgium', 'addendum': {'best': {'streetname_fr': 'Chaussée Brunehault', 'municipality_name_fr': 'Quiévrain', 'NIS': 53068, 'municipality_id': 'geodata.wallonie.be/id/Municipality/53068/7', 'street_id': 'geodata.wallonie.be/id/Streetname/7714480/2'}}}}, {'type': 'Feature', 'geometry': {'type': 'Point', 'coordinates': [3.879077, 50.365515]}, 'properties': {'id': 'geodata.wallonie.be/id/Streetname/7712327/1_fr', 'gid': 'be-wal:street:geodata.wallonie.be/id/Streetname/7712327/1_fr', 'layer': 'street', 'source': 'be-wal', 'source_id': 'geodata.wallonie.be/id/Streetname/7712327/1_fr', 'country_code': 'BE', 'name': 'Chaussée Brunehault, 7080 Frameries', 'street': 'Chaussée Brunehault', 'postalcode': '7080', 'confidence': 1, 'match_type': 'exact', 'accuracy': 'centroid', 'country': 'Belgium', 'country_gid': 'whosonfirst:country:85632997', 'country_a': 'BEL', 'macroregion': 'Wallonia', 'macroregion_gid': 'whosonfirst:macroregion:404227353', 'region': 'Hainaut', 'region_gid': 'whosonfirst:region:85681723', 'region_a': 'HT', 'county': 'Mons', 'county_gid': 'whosonfirst:county:102049933', 'county_a': 'MN', 'label': 'Chaussée Brunehault, 7080 Frameries, HT, Belgium', 'addendum': {'best': {'streetname_fr': 'Chaussée Brunehault', 'municipality_name_fr': 'Frameries', 'NIS': 53028, 'municipality_id': 'geodata.wallonie.be/id/Municipality/53028/8', 'street_id': 'geodata.wallonie.be/id/Streetname/7712327/1'}}}}], 'bbox': [3.691458, 50.280646, 5.539947, 50.71061]}


In [41]:
street_res["features"][0]["properties"]["postalcode"]

'7141'

In [44]:
list(filter(lambda f: f["properties"]["postalcode"] == "4453" if "postalcode" in f["properties"] else False,  street_res["features"]))

[]

In [20]:
res ={'took': 3, 'timed_out': False, '_shards': {'total': 1, 'successful': 1, 'skipped': 0, 'failed': 0}, 'hits': {'total': {'value': 3, 'relation': 'eq'}, 'max_score': 22.6566, 'hits': [{'_index': 'pelias', '_type': '_doc', '_id': 'whosonfirst:locality:101755111', '_score': 22.6566, '_source': {'center_point': {'lon': 4.707492, 'lat': 50.482595}, 'parent': {'country': ['Belgium'], 'country_id': ['85632997'], 'country_a': ['BEL'], 'country_source': [None], 'county': ['Arrondissement of Namur'], 'county_id': ['102049931'], 'county_a': [None], 'county_source': [None], 'locality': ['Spy'], 'locality_id': ['101755111'], 'locality_a': [None], 'locality_source': [None], 'macroregion': ['Waals Gewest'], 'macroregion_id': ['404227353'], 'macroregion_a': [None], 'macroregion_source': [None], 'region': ['Provincie Namen'], 'region_id': ['85681699'], 'region_a': ['NA'], 'region_source': [None]}, 'bounding_box': '{"min_lat":50.466075,"max_lat":50.49642,"min_lon":4.686728,"max_lon":4.72668}', 'name': {'default': 'Spy', 'he': 'ספי', 'ja': 'スパイ', 'es': 'Gruta de Spy', 'wa': 'Spî'}, 'addendum': {'concordances': '{"fct:id":"036c4706-8f76-11e1-848f-cfd5bf3ef515","gn:id":2786236,"gp:id":20098375,"qs_pg:id":1185177,"wd:id":"Q1260067","wk:page":"Spy, Belgium"}'}, 'source': 'whosonfirst', 'source_id': '101755111', 'layer': 'locality'}}, {'_index': 'pelias', '_type': '_doc', '_id': 'whosonfirst:locality:1125987599', '_score': 22.266727, '_source': {'center_point': {'lon': 4.70944, 'lat': 50.48143}, 'parent': {'country': ['Belgium'], 'country_id': ['85632997'], 'country_a': ['BEL'], 'country_source': [None], 'county': ['Arrondissement of Namur'], 'county_id': ['102049931'], 'county_a': [None], 'county_source': [None], 'locality': ['Sy'], 'locality_id': ['1125987599'], 'locality_a': [None], 'locality_source': [None], 'macroregion': ['Waals Gewest'], 'macroregion_id': ['404227353'], 'macroregion_a': [None], 'macroregion_source': [None], 'region': ['Provincie Namen'], 'region_id': ['85681699'], 'region_a': ['NA'], 'region_source': [None]}, 'bounding_box': '{"min_lat":50.46143,"max_lat":50.50143,"min_lon":4.68944,"max_lon":4.72944}', 'name': {'default': ['Sy', 'Spy'], 'he': 'ספי', 'ja': 'スパイ', 'es': 'Gruta de Spy', 'wa': 'Spî'}, 'addendum': {'concordances': '{"gn:id":2786236,"gp:id":20098375,"qs_pg:id":1185177,"wd:id":"Q1260067","qs:id":1185177}'}, 'source': 'whosonfirst', 'source_id': '1125987599', 'layer': 'locality'}}, {'_index': 'pelias', '_type': '_doc', '_id': 'be-wal:locality:geodata.wallonie.be/id/Municipality/92140/7_5190_fr_Spy', '_score': 21.198465, '_source': {'center_point': {'lon': 4.704764, 'lat': 50.479636}, 'parent': {'country': ['Belgium'], 'country_id': ['85632997'], 'country_a': ['BEL'], 'country_source': [None], 'macroregion': ['Wallonia'], 'macroregion_id': ['404227353'], 'macroregion_a': [None], 'macroregion_source': [None], 'region': ['Namur'], 'region_id': ['85681699'], 'region_a': ['NA'], 'region_source': [None], 'county': ['Namur'], 'county_id': ['102049931'], 'county_a': ['NM'], 'county_source': [None], 'locality': ['Spy'], 'locality_id': ['101755111'], 'locality_a': [None], 'locality_source': [None]}, 'name': {'default': '5190 Jemeppe-sur-Sambre (Spy)'}, 'address_parts': {'zip': '5190'}, 'addendum': {'best': '{"municipality_name_fr":"Jemeppe-sur-Sambre","part_of_municipality_name_fr":"Spy","NIS":92140,"municipality_id":"geodata.wallonie.be/id/Municipality/92140/7","postal_code":"5190"}'}, 'source': 'be-wal', 'source_id': 'geodata.wallonie.be/id/municipality/92140/7_5190_fr_spy', 'layer': 'locality'}}]}}

res["hits"]["hits"][2]["_source"]#["addendum"]#["best"]
# res

{'center_point': {'lon': 4.704764, 'lat': 50.479636},
 'parent': {'country': ['Belgium'],
  'country_id': ['85632997'],
  'country_a': ['BEL'],
  'country_source': [None],
  'macroregion': ['Wallonia'],
  'macroregion_id': ['404227353'],
  'macroregion_a': [None],
  'macroregion_source': [None],
  'region': ['Namur'],
  'region_id': ['85681699'],
  'region_a': ['NA'],
  'region_source': [None],
  'county': ['Namur'],
  'county_id': ['102049931'],
  'county_a': ['NM'],
  'county_source': [None],
  'locality': ['Spy'],
  'locality_id': ['101755111'],
  'locality_a': [None],
  'locality_source': [None]},
 'name': {'default': '5190 Jemeppe-sur-Sambre (Spy)'},
 'address_parts': {'zip': '5190'},
 'addendum': {'best': '{"municipality_name_fr":"Jemeppe-sur-Sambre","part_of_municipality_name_fr":"Spy","NIS":92140,"municipality_id":"geodata.wallonie.be/id/Municipality/92140/7","postal_code":"5190"}'},
 'source': 'be-wal',
 'source_id': 'geodata.wallonie.be/id/municipality/92140/7_5190_fr_spy',
 